In [159]:
import json
import pandas as pd
from typing import Dict, List, Any
import numpy as np

def extract_gtfs(file_path):
    json_data = Dict[str, Any]
    with open(file_path, 'r', encoding='utf-8') as f:
        json_data = json.load(f)

    normalized_data = []
    for poi in json_data['elements']:
        # Copy basic properties like type, id, lat, lon
        item = {k: v for k, v in poi.items() if k != 'tags'}
        if 'tags' in poi and isinstance(poi['tags'], dict):
            item.update(poi['tags'])

        normalized_data.append(item)

    df = pd.DataFrame(normalized_data)
    df = df.replace('nan', np.nan)

    if not df.empty:
        priority_cols = ['id', 'name', 'lat', 'lon', 'opening_hours', 'amenity', 'cuisine', 'wheelchair', 'toilets', 'access']
        existing_cols = [col for col in priority_cols if col in df.columns]
        other_cols = [col for col in df.columns if col not in existing_cols]
        df = df[existing_cols + other_cols]

    return df

file_path = '../data_collection/raw_data/raw_osm.json'
data_df = extract_gtfs(file_path)

In [217]:
data_df['amenity'].unique()

array([nan, 'fast_food', 'restaurant', 'cafe', 'post_office', 'fountain',
       'bench', 'theatre', 'clock', 'bicycle_parking', 'planetarium',
       'conference_centre', 'fire_station', 'cinema', 'toilets',
       'arts_centre', 'place_of_worship'], dtype=object)

## Important Accessibility related tags

* wheelchair
* toilets
* toilets:unisex
* toilets:wheelchair
* changing_table
* sensory_friendly:accommodation
* drive_through
* air_conditioning

## Rule-based Scheduler

In [284]:
import pandas as pd
from typing import List, Dict, Optional, Any
import math

class ItineraryScheduler:
    def __init__(self, data_df):
        self.data_df = data_df.copy().reset_index(drop=True)
        self.data_df['lat'] = pd.to_numeric(self.data_df['lat'], errors='coerce')
        self.data_df['lon'] = pd.to_numeric(self.data_df['lon'], errors='coerce')
        # self.data_df.dropna(subset=['lat', 'lon'], inplace=True)

    @staticmethod
    def calculate_distance(lat1, lon1, lat2, lon2):
        radius = 3958.8  #miles
        
        lat1_rad, lon1_rad = math.radians(lat1), math.radians(lon1)
        lat2_rad, lon2_rad = math.radians(lat2), math.radians(lon2)
        
        d_lat = lat2_rad - lat1_rad
        d_lon = lon2_rad - lon1_rad
        
        # Haversine formula
        a = math.sin(d_lat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(d_lon / 2)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        
        return radius * c
    
    def update_basic_score(self, df, accessibility_needs):
        df['base_score'] = 0
        helpful_tags = ['opening_hours', 'phone', 'website']
        for ht in helpful_tags:
            df.loc[df[ht].notna(), 'base_score'] += 1

        if accessibility_needs:
            df.loc[df['wheelchair'].isin(['yes', 'designated']), 'base_score'] += 2
            df.loc[df['toilets:wheelchair'].isin(['yes', 'designated']), 'base_score'] += 2
        
        return df

    def rule_based_decision(self, prefs, poi_type):
        pois = self.data_df.copy()
        pois = self.update_basic_score(pois, prefs['must_be_accessible'])

        # check for poi type
        if poi_type == 'amenity':
            pois = pois[
                pois['amenity'].str.contains(prefs['amenity_type'], case=False, na=False)
            ]
        elif poi_type == 'tourism':
            pois = pois[
                pois['tourism'].str.contains(prefs['tourism_type'], case=False, na=False)
            ]
        
        # check for cuisine
        if 'cuisine' in prefs:
            not_restuarant = (pois['amenity'] != 'restaurant')
            restaurant_with_cuisine = (
                (pois['amenity'] == 'restaurant') & 
                (pois['cuisine'].str.contains(prefs['cuisine'], case=False, na=False))
            )
            pois = pois[not_restuarant | restaurant_with_cuisine]

        # check for accesibility
        if prefs.get('must_be_accessible', False):
            pois = pois[
                pois['wheelchair'].isin(['yes', 'designated'])
            ]

        # Remove items already visited
        if 'visited_ids' in prefs:
            pois = pois[~pois['id'].isin(prefs['visited_ids'])]

        return pois

    def create_itinerary(self, num_pois, start_lat, start_lon, prefs, time_per_poi_hrs, max_travel_mi):        
        itinerary = []
        curr_lat, curr_lon = start_lat, start_lon
        
        travel_time = num_pois * 0.5 # estimate 30 min for moving
        total_estimated_time = (num_pois * time_per_poi_hrs) + travel_time
        
        # Limit possible time
        if total_estimated_time > 10:
             return [{"Error": "Requested trip is too ambitious for one day (" + str(total_estimated_time) + " hours needed). Reduce the number of POIs."}]

        for i in range(num_pois):
            poi_type = 'amenity' if i % 2 == 0 else 'tourism'
            pois = self.rule_based_decision(prefs, poi_type)
            
            if pois.empty:
                print(f"No more pois matching prefs.")
                break

            pois['dist'] = pois.apply(
                lambda row: self.calculate_distance(curr_lat, curr_lon, row['lat'], row['lon']),
                axis=1
            )

            # only consider poi within range limit
            pois = pois[pois['dist'] <= max_travel_mi]
            
            if pois.empty:
                print(f"Warning: Stopped at POI {i+1}. No nearby POIs found (max {max_travel_mi} mi).")
                break

            
            # Get final score and sort
            pois['final_score'] = pois['base_score'] - (pois['dist'] * 0.1) 
            best_choice = pois.sort_values(by='final_score', ascending=False).iloc[0]
            
            chosen_poi = {
                "name": best_choice['name'],
                "type": best_choice[poi_type],
                "dist_from_prev": round(best_choice['dist'], 2),
                "duration": f"{time_per_poi_hrs} hours",
                "score": round(best_choice['final_score'], 2),
                "location": f"({round(best_choice['lat'], 4)}, {round(best_choice['lon'], 4)})"
            }
            
            if chosen_poi['type'] == 'restuarant':
                chosen_poi["cuisine"] = best_choice.get('cuisine', 'N/A')

            itinerary.append(chosen_poi)

            # update
            curr_lat = best_choice['lat']
            curr_lon = best_choice['lon']
            prefs['visited_ids'].add(best_choice['id'])

        return itinerary

## Run Rule-Based Scheduler

In [286]:
# User-inputted variables
START_LAT = 32.7106
START_LON = -117.1626 
NUM_POIS_TO_VISIT = 4
TIME_PER_POI = 2
MAX_DIST = 20

# User Preferences
USER_PREFERNCES = {
    "amenity_type": "restaurant|theatre|cafe", #'fast_food', 'restaurant', 'cafe', 'post_office', 'fountain', 'bench', 'theatre', 'clock', 'bicycle_parking', 'planetarium', 'conference_centre', 'fire_station', 'cinema', 'toilets', 'arts_centre', 'place_of_worship'
    "tourism_type": "museum|gallery|viewpoint|attraction|aquarium", # 'attraction', 'gallery', 'museum', 'viewpoint', 'artwork', 'aquarium', 'zoo', 'theme_park'
    "cuisine": "italian|mexican|american",
    "must_be_accessible": True,
    "visited_ids": set()
}

# Scheduler
scheduler = ItineraryScheduler(data_df)
print(f"Starting Location: ({START_LAT}, {START_LON})")
print(f"Generating One day Itinerary for {NUM_POIS_TO_VISIT} Points of Interest...")
print("-" * 80)

itinerary = scheduler.create_itinerary(
    num_pois=NUM_POIS_TO_VISIT, 
    start_lat=START_LAT, 
    start_lon=START_LON, 
    prefs=USER_PREFERNCES,
    time_per_poi_hrs=TIME_PER_POI,
    max_travel_mi=MAX_DIST,
)

# Result
for i, poi in enumerate(itinerary):
    if i == 0:
        dist = f"Start: ({round(START_LAT, 4)}, {round(START_LON, 4)})"
    else:
        dist = f"From Previous: {poi['dist_from_prev']} mi"

    print(f"POI {i + 1}: {poi['name']} ({poi['type']})")
    print(f"  - Location: {poi['location']} ({dist})")
    print(f"  - Duration: {poi['duration']}")
    print(f"  - Final Score: {poi['score']}\n")

Starting Location: (32.7106, -117.1626)
Generating One day Itinerary for 4 Points of Interest...
--------------------------------------------------------------------------------
POI 1: Tiger Coffee (cafe)
  - Location: (32.7113, -117.1612) (Start: (32.7106, -117.1626))
  - Duration: 2 hours
  - Final Score: 5.99

POI 2: San Diego History Center (museum)
  - Location: (32.7312, -117.1483) (From Previous: 1.56 mi)
  - Duration: 2 hours
  - Final Score: 3.84

POI 3: Modern Times Coffee (cafe)
  - Location: (32.7118, -117.1563) (From Previous: 1.41 mi)
  - Duration: 2 hours
  - Final Score: 4.86



## TODO:
* need to work on evaluation metrics
* need more data for accessibility because it is currently too limited